# Workshop: Introducción a Generative AI en Oracle y Creación de Agentes con LangChain

Bienvenidos al workshop. En esta sesión vamos a explorar cómo usar los **servicios de IA generativa de Oracle** para resolver problemas reales y luego **crear un agente inteligente** usando **LangChain** que pueda interactuar con estos servicios.

## Objetivos de la sesión
- Conocer la oferta de **Oracle Cloud Infrastructure (OCI Generative AI)** y cómo integrarla desde Python.
- Ejecutar peticiones a modelos de lenguaje para **generar texto** de manera controlada.
- Construir un **agente con LangChain** que use herramientas (como SQL o RAG) para responder preguntas de forma autónoma.
- Aprender buenas prácticas para **orquestar flujos de trabajo** y extender capacidades de los modelos.

## Requisitos previos
- Conocimientos básicos de Python 🐍
- Tener acceso a una cuenta de **Oracle Cloud** con permisos para usar **OCI Generative AI**
- Familiaridad básica con entornos virtuales y Jupyter Notebooks.

> 💡 **Tip:** este notebook está diseñado para ser práctico y paso a paso. Podrás copiar, ejecutar y modificar el código para experimentar con los conceptos que vamos a explicar.

¡Vamos a empezar!

### A continuación... 

📰 Recopilaremos noticias sobre el paro del 16 de Septiembre ocurrido en la ciudad de Bogotá 

🤖 Consumiremos un modelo de lenguaje alojado en Oracle Cloud 

🔍 Construiremos un agente con langchain que es capaz de responder a preguntas relacionadas con el paro del 16 de septiembre 

## Notebook

### Instalación

In [1]:
!pip install -r requirements.txt

### Importación de librerías

In [ ]:
import json
import re
from typing import Dict, List

import oci
import requests
from IPython.display import Markdown, display
from langchain.agents import AgentExecutor, create_react_agent
from langchain_community.chat_models import ChatOCIGenAI
from langchain_core.messages import HumanMessage
from langchain_core.prompts import PromptTemplate
from langchain_core.tools import tool
from oci.auth.signers import get_resource_principals_signer
from oci.config import from_file
from tavily import TavilyClient
from oci.generative_ai import GenerativeAiClient
from dotenv import load_dotenv
import os
from database import DatabaseClient
from langchain_community.embeddings import OCIGenAIEmbeddings



### Registro y configuración de Tavily

 Nuestro agente necesita acceso a la web para acceder a las últimas noticias del tema que le hemos indicado, para esto, es necesario configurar una herramienta que le permita a nuestro agente navegar por portales de noticias, por eso usaremos [Tavily](https://www.tavily.com/).

#### 🪪 Registro en Tavily (paso a paso)

1) Abre **https://app.tavily.com/home** y haz clic en **Sign Up**. Verifica tu correo electrónico para activar la cuenta. ![image1](./images/tavily_signup.png)
2) Inicia sesión: en la **página principal** verás tu **API Key**. Haz **Copy** para copiarla. ![image2](./images/api_key.png)


In [ ]:
# Pega la API Key de Tavily aquí
TAVILY_API_KEY = ""

In [4]:
# No ajustes el codigo esta es solo para validar si la llave se configuro como una variable
try:
    assert TAVILY_API_KEY != "Pega aqui tu llave de Tavily", "Por favor, pega tu API Key de Tavily en la variable TAVILY_API_KEY"
    print("✅ API Key de Tavily configurada correctamente")
except AssertionError as e:
    print(f"❌ Error: {e}")
    print("⚠️ Debes configurar tu API Key de Tavily en la celda anterior")

✅ API Key de Tavily configurada correctamente


El siguiente codigo como se denota crea una lista con las paginas oficiales del pais para poder con tavily extraer informacion relevante de acuerdo al topico de noticia a buscar

In [5]:

DOMAINS = [
    "www.bloomberglinea.com",
    "www.semana.com",
    "es-us.noticias.yahoo.com",
    # Agrega más dominios relevantes aquí
]
@tool
def search_in_internet(pregunta:str="¿Qué es la inflación?") -> str:
    """Busca información en internet. Usa esta herramienta para buscar información actualizada en internet que no esté en tu base de conocimientos."""

    client = TavilyClient(TAVILY_API_KEY)
    response = client.search(
    query=pregunta,
    include_domains=DOMAINS,
    #topic="news",
    #days=45,
    #max_results=10
    )
    return response

## Configuración de la autenticación del SDK de OCI

Desde este notebook es necesario acceder a algunos servicios de Oracle, como el servicio de Generative AI, aunque ejecutes este notebook en cloud o de forma local, es necesario configurar las credenciales en la máquina que realiza el consumo del servicio. 

### Configuración de credenciales en Data Science

Ejecuta las siguientes celdas para la creación de la configuración en un ambiente de Data Science

In [6]:
# crea carpeta y permisos
!mkdir -p /home/datascience/.oci

mkdir: /home/datascience: Operation not supported


In [7]:
# Ver tu HOME y listar (incluye ocultos)
# La carpeta donde esta el home en el servicio es /home/datascience
!echo $HOME
!ls -la $HOME | head -n 30

/Users/valefeve
total 228928
drwxr-x---@  84 valefeve  staff      2688 Feb  9 08:30 .
drwxr-xr-x    5 root      admin       160 Jan 12 19:08 ..
-r--------    1 valefeve  staff         7 Sep  7 15:11 .CFUserTextEncoding
-rw-r--r--@   1 valefeve  staff     18436 Feb  8 20:24 .DS_Store
drwx------+   2 valefeve  staff        64 Feb  5 15:59 .Trash
drwxr-xr-x@   3 valefeve  staff        96 Nov  4 14:52 .anaconda_backup
-rw-r--r--    1 valefeve  staff       618 Sep  8 07:16 .anyconnect
-rw-r--r--@   1 valefeve  staff       595 Jan 29 09:54 .bash_profile
-rw-r--r--@   1 valefeve  staff       144 Jan 29 09:54 .bashrc
drwxr-xr-x@   8 valefeve  staff       256 Dec 27 14:50 .cache
drwxr-xr-x    5 valefeve  staff       160 Sep  8 07:52 .cisco
drwx------@   9 valefeve  staff       288 Oct 29 09:42 .claude
-rw-------@   1 valefeve  staff     36522 Oct 29 09:42 .claude.json
-rw-------@   1 valefeve  staff     36076 Oct 29 09:42 .claude.json.backup
drwxr-xr-x@  18 valefeve  staff       576 Oct 25 22:1

In [ ]:
# Se crea una carpeta .oci donde guarda la llave y el archivo de configuracion
!mkdir -p ~/.oci
!ls -la ~/.oci


Para obtener la configuración y el api key, es necesario seguir los siguientes pasos [Configurar credenciales en ambiente local](../utils/Configurar%20credenciales%20en%20ambiente%20local.md). Este tutorial explica cómo obtener la configuración, las credenciales y en el paso 5 se detalla cómo realizar la configuración en el ambiente local. 

Para realizar la configuración en Data Science y no en un ambiente local, reemplazaremos el paso 5 por el siguiente.

Ahora vamos a ubicar la llave privada que descargamos en los pasos anteriores al generar el API Key. El archivo tendrá un nombre similar a “tu_usuario-año-mes-diaTHH_MM_SS.XXX.pem”.

Este archivo debe renombrarse como **“oci_api_key.pem”** y cargarse en Data Science utilizando la opción “Upload Files”, o bien arrastrándolo directamente en el menú izquierdo del navegador.

Una vez que el archivo esté cargado, podremos proceder con la ejecución de la siguiente línea.

In [ ]:
# Este codigo de linux mueve la llave a la carpeta .oci y le cambia los permisos
!mv ~/oci_api_key.pem ~/.oci/oci_api_key.pem
!chmod 600 ~/.oci/oci_api_key.pem
!ls -la ~/.oci

A continuación, crearemos el archivo de configuración en la ruta ~/.oci/config, vamos a copiar los valores de la configuración mostrada en pantalla y a reemplazarlos en la siguiente línea.


Reemplazaremos **_ocid1.user.oc1.._** por el ocid del usuario mostrado en pantalla<br>
Reemplazaremos **_fingerprint_** por el figerprint mostrado en pantalla<br>
Reemplazaremos **_tenancy_** por el figerprint mostrado en pantalla<br>
Reemplazaremos **_region_** por el figerprint mostrado en pantalla<br>
🚨 No reemplazaremos **_key_file_** por ninguna ruta si estamos ejecutando este notebook en DataScience. Si queremos ejecutar este notebook de forma local, podemos reemplazar la ruta por ~/.oci/nombre_de_la_key.pem 

In [ ]:
%%bash
cat > ~/.oci/config <<'CFG'
[DEFAULT]
user=ocid1.user.oc1.....
fingerprint=4a:c7:6f:.....
tenancy=ocid1.tenancy.oc1......
region=region de tu tenant!
key_file=/home/datascience/.oci/oci_api_key.pem
CFG

echo "Config creado en ~/.oci/config"
cat ~/.oci/config | sed 's/fingerprint=.*/fingerprint=<oculto>/'

In [ ]:
# Quita posibles finales de línea de Windows (CRLF)
!sed -i 's/\r$//' ~/.oci/config

# mostrar
!sed -n '1,200p' ~/.oci/config

### Configuración de credenciales en un ambiente local

Para realizar la configuración en el ambiente local, siga los siguientes pasos [Configurar credenciales en ambiente local](../utils/Configurar%20credenciales%20en%20ambiente%20local.md).

In [8]:
#carga a memoria la configuracion de OCI para enviarla al API de Generative AI
config = from_file()

In [ ]:
# Descomenta únicamente la línea que corresponda a tu región en la cual esta tu tenant
#REGION = "sa-saopaulo-1"
REGION = "us-chicago-1"
#REGION = "uk-london-1"
#REGION = "eu-frankfurt-1"
#REGION = "ap-osaka-1"
#REGION = "us-ashburn-1"

match REGION:
    case "us-chicago-1":
        MODEL_OCID = "ocid1.generativeaimodel.oc1.us-chicago-1.amaaaaaask7dceyayjawvuonfkw2ua4bob4rlnnlhs522pafbglivtwlfzta"
        MODEL_EMBED_OCID="cohere.embed-multilingual-v3.0"
    case "sa-saopaulo-1":
        MODEL_OCID = "ocid1.generativeaimodel.oc1.sa-saopaulo-1.amaaaaaask7dceyarsn4m6k3aqvvgatida3omyprlcs3alrwcuusblru4jaa"
        MODEL_EMBED_OCID="cohere.embed-multilingual-v3.0"
    case "uk-london-1":
        MODEL_OCID = "ocid1.generativeaimodel.oc1.uk-london-1.amaaaaaask7dceyach2dyu6g5w5ocnvbkto2g76wxitj3rpddplsqoxqh2lq"
        MODEL_EMBED_OCID="cohere.embed-multilingual-v3.0"
    case "eu-frankfurt-1":
        MODEL_OCID = "ocid1.generativeaimodel.oc1.eu-frankfurt-1.amaaaaaask7dceya4tdabclcsqbc3yj2mozvvqoq5ccmliv3354hfu3mx6bq"
        MODEL_EMBED_OCID="cohere.embed-multilingual-v3.0"
    case "ap-osaka-1":
        MODEL_OCID = "ocid1.generativeaimodel.oc1.ap-osaka-1.amaaaaaask7dceyaei4b6vhy7rgsqzuqet4ndnnun4fhxco2tfzusslg35wa"
        MODEL_EMBED_OCID="cohere.embed-multilingual-v3.0"
    case "us-ashburn-1":
        MODEL_OCID = "ocid1.generativeaimodel.oc1.iad.amaaaaaask7dceyah6tjdejjashngznsylutuhhvufukzb2g2ls54g2flsfq"
        MODEL_EMBED_OCID="cohere.embed-multilingual-v3.0"


In [10]:
# No ajustes este bloque de código. Este bloque es solo para validar si la region se configuró como una variable
assert REGION in ["sa-saopaulo-1", "us-chicago-1", "uk-london-1", "eu-frankfurt-1", "ap-osaka-1"], "Por favor, descomenta la línea que corresponda a tu región"

In [ ]:
# Aquí debes pegar el OCID de tu compartimento
# Encuentra el OCID de tu compartimento en la consola de Oracle Cloud, en la sección de Compartments https://cloud.oracle.com/identity/compartments
COMPARTMENT_ID = ""

In [12]:
SERVICE_ENDPOINT = f"https://inference.generativeai.{REGION}.oci.oraclecloud.com"


In [ ]:
genai = GenerativeAiClient(config=config)
models = genai.list_models(
    compartment_id=COMPARTMENT_ID,
    capability=["CHAT"],
    lifecycle_state="ACTIVE"
).data.items
assert models, "No hay modelos CHAT visibles en el compartimento. Revisa permisos/compartimento."

print("Modelos CHAT disponibles:")
for model in models:
    print(f"{model.display_name} by {model.vendor} - ocid: {model.id}")

Modelos CHAT disponibles:
cohere.command-a-vision by cohere - ocid: ocid1.generativeaimodel.oc1.us-chicago-1.amaaaaaask7dceya3osthuanpx3xb7c6hc3blqoa76b3cesvce7pdhkmbaqq
cohere.command-a-reasoning by cohere - ocid: ocid1.generativeaimodel.oc1.us-chicago-1.amaaaaaask7dceyagw2uzvswckq7snfu2kbrq2iw4ul4rzwlifeessuyunfq
xai.grok-4-1-fast-reasoning by xai - ocid: ocid1.generativeaimodel.oc1.us-chicago-1.amaaaaaask7dceyadd6ow2hxfppx7dmwmok4pon2jtsw2m2wiwoplexjrqaq
xai.grok-4-1-fast-non-reasoning by xai - ocid: ocid1.generativeaimodel.oc1.us-chicago-1.amaaaaaask7dceyaurmacricgczrbvmgd5z2b5f23lhntjbqdq7axq63rcuq
openai.gpt-5.2-pro-2025-12-11 by openai - ocid: ocid1.generativeaimodel.oc1.us-chicago-1.amaaaaaask7dceyaoufxtdrpz27aevjdhwx7t2y7crqdoeczbvlb7lglbibq
openai.gpt-audio by openai - ocid: ocid1.generativeaimodel.oc1.us-chicago-1.amaaaaaask7dceyav4ahdq7uzdfvi3uaxirxnno5hawvwzwrotczwkv4phkq
openai.gpt-5.1-chat-latest by openai - ocid: ocid1.generativeaimodel.oc1.us-chicago-1.amaaaaaask7dceya

In [14]:
# Aquí se crea el cliente de inferencia
inf = oci.generative_ai_inference.GenerativeAiInferenceClient(
    config=config,
    service_endpoint=SERVICE_ENDPOINT
)

# Este prompt corresponde a la pregunta que se le hará al modelo
user_input = "¿Qué es Bre-B y cómo afecta las plataformas financieras en Colombia?"

# Aquí se construye el request
content = oci.generative_ai_inference.models.TextContent(text=user_input)
message = oci.generative_ai_inference.models.Message(role="USER", content=[content])

chat_request = oci.generative_ai_inference.models.GenericChatRequest(
    api_format=oci.generative_ai_inference.models.BaseChatRequest.API_FORMAT_GENERIC,
    messages=[message],
    max_tokens=20,
    temperature=0.1,
    frequency_penalty=0.0,
    presence_penalty=0.0,
    seed=42
)

chat_detail = oci.generative_ai_inference.models.ChatDetails(
    serving_mode=oci.generative_ai_inference.models.OnDemandServingMode(model_id=MODEL_OCID),
    chat_request=chat_request,
    compartment_id=COMPARTMENT_ID,
)

# Aquí se realiza la llamada al modelo
resp = inf.chat(chat_detail)

# Aquí se procesa el resultado
choices = resp.data.chat_response.choices
response_text = choices[0].message.content[0].text if choices else "No se generó respuesta."
print(json.dumps({"response": response_text}, indent=2, ensure_ascii=False))

{
  "response": "Bre-B, o \"Breach-B\", se refiere a un tipo de ataque cibernético dirigido"
}


### ❓ Preguntas

- ¿cuál es tu región?
- ¿cuántos modelos hay en tu región?
- ¿qué parámetro del modelo define el tamaño de la respuesta?
- Haz un análisis de la pregunta y la respuesta del modelo

## 🤖 Creación del Agente LangChain

In [ ]:
# Configura tu endpoint y compartimento
#ENDPOINT = f"https://inference.generativeai.{REGION}.oci.oraclecloud.com"

llm = ChatOCIGenAI(
  model_id=MODEL_EMBED_OCID,
  service_endpoint=SERVICE_ENDPOINT,
  compartment_id=COMPARTMENT_ID,
  provider="meta",
  model_kwargs={
    "temperature": 0.3, 
    "max_tokens": 800,   
    "top_p": 0.8,  
    "frequency_penalty": 0,
    "presence_penalty": 0,
  },
  auth_type="API_KEY",
  auth_profile="DEFAULT"
)

tools = [
    # Aquí se deben agregar las herramientas que se quieran usar en el agente, por ejemplo: search_in_internet
]

react_prompt_template = """

Eres un asistente inteligente diseñado para ayudar a los usuarios a resolver preguntas y tareas de forma clara, útil y confiable, independientemente del dominio (noticias, finanzas, tecnología, educación, salud, u otros).

Tu objetivo es comprender la intención del usuario, razonar sobre el problema y entregar una respuesta clara.

Herramientas disponibles:
{tools}

Usa EXACTAMENTE este formato:

Question: la pregunta a responder
Thought: explica qué harás
Action: una de [{tool_names}]
Action Input: el input para la acción (o "" si no aplica)
Observation: resultado de la acción
Thought: analiza y sintetiza
Final Answer: respuesta clara y útil en español 

Comienza.

Question: {input}
Thought: {agent_scratchpad}"""

prompt = PromptTemplate.from_template(react_prompt_template)
agent = create_react_agent(llm, tools, prompt)

agent_executor = AgentExecutor(
    agent=agent,
    tools=tools,
    verbose=True,
    max_iterations=20,
    stream_runnable=False,
    handle_parsing_errors=True
)
pregunta = "¿Qué es Bre-B y cómo afecta las plataformas financieras en Colombia?"
respuesta = agent_executor.invoke({"input": pregunta})




> Entering new AgentExecutor chain...
Para responder a esta pregunta, primero debo entender qué es Bre-B y luego analizar su impacto en las plataformas financieras en Colombia.

Action: Buscar información sobre Bre-B
Action Input: "Bre-B Colombia"Buscar información sobre Bre-B is not a valid tool, try one of [].Dado que no tengo acceso a una herramienta de búsqueda directa, puedo intentar deducir o proporcionar información general sobre el tema. Sin embargo, si "Bre-B" se refiere a un término específico relacionado con finanzas o tecnología en Colombia, podría tratarse de una entidad, un concepto o una tecnología particular.

Action: Proporcionar una respuesta basada en deducciones y conocimientos generales
Action Input: ""Proporcionar una respuesta basada en deducciones y conocimientos generales is not a valid tool, try one of [].Question: ¿Qué es Bre-B y cómo afecta las plataformas financieras en Colombia?

Thought: Para responder a esta pregunta, primero debo entender qué es Bre-B

In [16]:
respuesta

{'input': '¿Qué es Bre-B y cómo afecta las plataformas financieras en Colombia?',
 'output': 'Sin información específica sobre "Bre-B", es difícil determinar su impacto exacto en las plataformas financieras en Colombia. Sin embargo, si se tratara de una innovación o una nueva entidad financiera, podría potencialmente aumentar la competencia, impulsar la innovación y mejorar los servicios financieros disponibles en el país.'}

¿Qué función cumple la herramienta (o tool) que se incluyó en el agente?
¿Cuál es la diferencia entre la respuesta del modelo anterior con generative ai y la respuesta del agente con una tool?

In [17]:
# 1) Toma la salida del agente y conviértela a texto de forma segura
def _to_text(x):
    if isinstance(x, dict):
        # LangChain AgentExecutor suele devolver {"output": "..."}
        return x.get("output") or json.dumps(x, ensure_ascii=False, indent=2)
    return str(x)
insumos = _to_text(respuesta)   # <--- usa la variable 'respuesta' que ya tienes del agente
# 2) Prompt de análisis/síntesis para Fiestas Patrias (Chile)
analysis_prompt = f"""
Eres un asistente analítico cuyo objetivo es comprender información proporcionada y transformarla en una respuesta clara, estructurada y útil para el usuario.

Responde exclusivamente en español.

A continuación se entregan los únicos insumos que puedes utilizar.
Pueden contener texto libre, listas, tablas o estructuras como JSON, y pueden estar incompletos.
{insumos}
---
### TAREAS

Responde de forma directa y clara a la pregunta planteada.

Extrae y organiza la información relevante solo si está disponible en los insumos, priorizando lo más importante.

Identifica hechos, acciones, fechas, impactos, actores o medidas relevantes cuando existan.

Si hay diferencias o inconsistencias entre los insumos, menciónalas brevemente y de forma neutral.

Si falta información clave, indícalo explícitamente.
---
### FORMATO (Markdown)
## Resumen
Síntesis breve (3–5 líneas) con los puntos más relevantes para entender la situación.

## Detalles relevantes (si hay información)
Elemento / tema: descripción clara
Información clave
Condiciones o contexto
Observaciones relevantes
Enlace o referencia (si existe)

## Recomendaciones o acciones (si hay información)
Lista clara y concreta de acciones, sugerencias o conclusiones derivadas de los insumos.

## Fuentes
Lista de referencias o enlaces solo si aparecen explícitamente en los insumos.
Si no hay referencias, escribe: No disponible.
---
### REGLAS
-  No inventes datos, cifras, fechas, actores ni enlaces.
- Si una sección no tiene información suficiente, indica: No disponible.
- No hagas suposiciones ni completes vacíos con conocimiento previo.
- Mantén un tono claro, neutral y orientado a facilitar la comprensión.
"""
# 3) Invoca la LLM (no streaming) y muestra en Markdown
analysis_response = llm.invoke(analysis_prompt)
display(Markdown(analysis_response.content))


## Resumen
La información disponible sobre "Bre-B" es limitada, lo que dificulta determinar su impacto en las plataformas financieras en Colombia. Sin embargo, si "Bre-B" se refiere a una innovación o nueva entidad financiera, podría potencialmente aumentar la competencia, impulsar la innovación y mejorar los servicios financieros en el país.

## Detalles relevantes
* Elemento / tema: Impacto de "Bre-B" en las plataformas financieras en Colombia
* Información clave: No hay información específica disponible sobre "Bre-B".
* Condiciones o contexto: Se asume que "Bre-B" podría ser una innovación o una nueva entidad financiera.
* Observaciones relevantes: El impacto potencial mencionado incluye aumento de la competencia, impulso a la innovación y mejora en los servicios financieros.

## Recomendaciones o acciones
* Buscar información adicional sobre "Bre-B" para entender su naturaleza y propósito.
* Analizar el contexto financiero actual en Colombia para evaluar posibles impactos.
* Considerar estudios o análisis de casos similares de innovaciones o nuevas entidades financieras en el país.

## Fuentes
No disponible.

### ❓ Preguntas
- ¿Qué sucede si agrego una quinta tarea indicandole al agente que debe hacer una predicción futurista?
- ¿El modelo conoce los detalles de la conexión a la base de conocimiento?
- ¿El modelo comprende la implementación de las herramientas que posee?



Qué puedes hacer para que el modelo evite responder información de sus herramientas internas?

In [19]:
load_dotenv()  # Carga las variables de entorno desde el archivo .env

True

In [ ]:
db_client = DatabaseClient()

In [ ]:
embeddings_client = OCIGenAIEmbeddings(
    model_id=MODEL_EMBED_OCID,
    service_endpoint=SERVICE_ENDPOINT,
    compartment_id=COMPARTMENT_ID,
)

def get_embedding(text: str) -> List[float]:
    embedding = embeddings_client.embed_query(text)
    return embedding

In [42]:
@tool
def search_information_in_knowledgebase(query: str, threshold: float = 0.8, max_results: int = 5) -> str:
    """"
    Search for relevant information in the knowledgebase using vector embeddings. This tool will search for the most similar documents based on the provided query.
    Not necesarily the most relevant, just the most similar in terms of vector distance. That's why it's important to perform multiple searches if needed, using different phrasings.
    """
    sql = open("./sqls/query_similar.sql").read()
    embeddings = get_embedding(query)
    print(embeddings)
    embedding_as_str = '[' + ', '.join(f'{x:.6f}' for x in embeddings) + ']'
    results = db_client.execute_query(sql, params={"embedding": embedding_as_str, "max_results": max_results})
    results_in_markdown = "\n".join([f"{row}" for row in results])
    return results_in_markdown


In [43]:
search_information_in_knowledgebase("an engineer")

[0.021057129, 0.0074501038, -0.03237915, 0.05734253, -0.022125244, 0.005718231, 0.013572693, -0.008132935, -0.0154800415, 0.02519226, 0.024459839, 0.039031982, 0.012512207, 0.01876831, 0.033477783, 0.016036987, 0.03491211, 0.0138168335, 0.024795532, -0.034362793, -0.02368164, 0.0023765564, 0.041290283, 0.0003311634, 0.010269165, 0.033721924, 0.0209198, -0.013046265, 0.06048584, -0.009025574, 0.0051078796, -0.027862549, 0.061950684, 0.0037269592, -0.02810669, -0.030853271, -0.016983032, -0.037353516, -0.010757446, 0.05831909, -0.05380249, 0.004764557, -0.033233643, 0.02607727, -0.0769043, -0.03491211, 0.050933838, 0.058898926, -0.028274536, 0.062286377, 0.002128601, -0.029800415, 0.030059814, 0.010658264, 0.038635254, -0.006790161, 0.03427124, 0.012176514, -0.0022392273, 0.0096206665, 0.007457733, -0.029190063, -0.040985107, 0.025970459, -0.0042495728, 0.054382324, 0.033721924, 0.035491943, 0.028808594, 0.021240234, 0.0076065063, 0.043273926, 0.041137695, 0.013519287, -0.0054092407, -0.

"{'persona': 'An electronics engineer specializing in circuit design and components, particularly resistors.', 'cosine_distance': 0.3921210967467029}\n{'persona': 'An electrical engineer focused on industrial safety and electrical system design, likely with experience in power distribution and transformer installations.', 'cosine_distance': 0.40616370044108197}\n{'persona': 'A robotics engineer or researcher with a strong interest in miniature robotics and alternative energy sources, likely involved in the development of innovative mobile systems for search and rescue or space exploration.', 'cosine_distance': 0.4291647613387598}\n{'persona': 'A mechanical engineer specializing in hydraulic systems and piping design, with an emphasis on corrosion prevention and materials science.', 'cosine_distance': 0.43312895357331704}\n{'persona': 'A data center professional or IT infrastructure manager interested in precision and clarity of technical terminology.', 'cosine_distance': 0.472733562037